# 优化（Optimization）

对应课程：`phases/01-math-foundations/08-optimization`

> 训练神经网络，本质上就是找到谷底。

本 notebook 把 `optimizers.py` 里的核心函数拆开：每个函数一组中文注释，后面跟一小段可运行实验。完整打印型 demo 仍在 `optimizers.py`。

**贯穿全课的模式：** 梯度指出最陡上升方向；优化器决定沿反方向怎么迈步。学习率太大就飞出去，太小就爬行。


## 0. 依赖


In [1]:
import math


## 1. Rosenbrock：狭窄弯曲的谷

$$
f(x,y)=(1-x)^2+100(y-x^2)^2
$$

全局最小在 $(1,1)$，$f=0$。沿抛物线 $y=x^2$ 是浅谷，垂直方向很陡，所以原始梯度下降容易横跳、沿谷底爬得很慢。

$$
\frac{\partial f}{\partial x}=-2(1-x)-400x(y-x^2),\qquad
\frac{\partial f}{\partial y}=200(y-x^2)
$$


In [2]:
def rosenbrock(params):
    """香蕉谷。最小点 (1,1)。"""
    x, y = params
    return (1 - x) ** 2 + 100 * (y - x ** 2) ** 2


def rosenbrock_gradient(params):
    """解析梯度，供优化器 step。"""
    x, y = params
    df_dx = -2 * (1 - x) + 200 * (y - x ** 2) * (-2 * x)
    df_dy = 200 * (y - x ** 2)
    return [df_dx, df_dy]


start = [-1.2, 1.0]
print("f(-1.2, 1.0) =", round(rosenbrock(start), 4))
print("grad =", [round(g, 4) for g in rosenbrock_gradient(start)])
print("f(1,1) =", rosenbrock([1.0, 1.0]))


f(-1.2, 1.0) = 24.2
grad = [-215.6, -88.0]
f(1,1) = 0.0


## 2. 原始梯度下降

$$
\theta \leftarrow \theta - \eta\,\nabla f(\theta)
$$

每维用同一个学习率。Rosenbrock 上 $\eta$ 稍大就会沿陡壁发散。


In [3]:
class GradientDescent:
    def __init__(self, lr=0.001):
        self.lr = lr

    def step(self, params, grads):
        """每维独立：新参数 = 旧参数 - lr * 梯度。"""
        return [p - self.lr * g for p, g in zip(params, grads)]


gd = GradientDescent(lr=0.0005)
p = [-1.2, 1.0]
p = gd.step(p, rosenbrock_gradient(p))
print("one GD step ->", [round(x, 6) for x in p], "f =", round(rosenbrock(p), 4))


one GD step -> [-1.0922, 1.044] f = 6.5944


## 3. 动量 SGD

$$
v \leftarrow \mu v + \nabla f,\qquad
\theta \leftarrow \theta - \eta v
$$

速度项把过去梯度指数平均，沿浅谷能积蓄方向、少在陡壁上振荡。


In [4]:
class SGDMomentum:
    def __init__(self, lr=0.001, momentum=0.9):
        self.lr = lr
        self.momentum = momentum
        self.velocity = None

    def step(self, params, grads):
        if self.velocity is None:
            self.velocity = [0.0] * len(params)
        self.velocity = [
            self.momentum * v + g
            for v, g in zip(self.velocity, grads)
        ]
        return [p - self.lr * v for p, v in zip(params, self.velocity)]


mom = SGDMomentum(lr=0.0001, momentum=0.9)
p = [-1.2, 1.0]
p = mom.step(p, rosenbrock_gradient(p))
print("one momentum step ->", [round(x, 6) for x in p], "f =", round(rosenbrock(p), 4))


one momentum step -> [-1.17844, 1.0088] f = 19.1796


## 4. Adam：一阶矩 + 二阶矩，再偏差修正

$$
m_t=\beta_1 m_{t-1}+(1-\beta_1)g_t,\quad
v_t=\beta_2 v_{t-1}+(1-\beta_2)g_t^2
$$

$$
\hat m=\frac{m_t}{1-\beta_1^t},\quad
\hat v=\frac{v_t}{1-\beta_2^t},\quad
\theta\leftarrow\theta-\eta\frac{\hat m}{\sqrt{\hat v}+\varepsilon}
$$

$m$ 是带动量的方向，$v$ 按坐标把学习率除以 RMS 梯度。前几步 $\beta^t$ 还没衰减，必须做偏差修正，否则 $m,v$ 被初始化 0 拉偏。


In [5]:
class Adam:
    def __init__(self, lr=0.001, beta1=0.9, beta2=0.999, epsilon=1e-8):
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.epsilon = epsilon
        self.m = None
        self.v = None
        self.t = 0

    def step(self, params, grads):
        if self.m is None:
            self.m = [0.0] * len(params)
            self.v = [0.0] * len(params)

        self.t += 1
        self.m = [
            self.beta1 * m + (1 - self.beta1) * g
            for m, g in zip(self.m, grads)
        ]
        self.v = [
            self.beta2 * v + (1 - self.beta2) * g ** 2
            for v, g in zip(self.v, grads)
        ]
        m_hat = [m / (1 - self.beta1 ** self.t) for m in self.m]
        v_hat = [v / (1 - self.beta2 ** self.t) for v in self.v]
        return [
            p - self.lr * mh / (vh ** 0.5 + self.epsilon)
            for p, mh, vh in zip(params, m_hat, v_hat)
        ]


adam = Adam(lr=0.01)
p = [-1.2, 1.0]
p = adam.step(p, rosenbrock_gradient(p))
print("one Adam step ->", [round(x, 6) for x in p], "f =", round(rosenbrock(p), 4))


one Adam step -> [-1.19, 1.01] f = 21.2878


## 5. `optimize` 与到最小点的距离

循环：求梯度 → `optimizer.step`。笔记本里步数压到几百，避免 5000 步刷屏。遇到 NaN / Inf / 爆炸就提前停。


In [6]:
def optimize(optimizer, func, grad_func, start, steps=400):
    """返回参数轨迹（含起点）。默认 400 步，比源文件的 5000 轻。"""
    params = list(start)
    history = [params[:]]
    for _ in range(steps):
        try:
            grads = grad_func(params)
            if any(math.isnan(g) or math.isinf(g) or abs(g) > 1e15 for g in grads):
                break
            params = optimizer.step(params, grads)
            if any(math.isnan(p) or math.isinf(p) or abs(p) > 1e15 for p in params):
                break
            history.append(params[:])
        except (OverflowError, ValueError):
            break
    return history


def distance_to_minimum(params, target=(1.0, 1.0)):
    """欧氏距离到 (1,1)。"""
    return math.sqrt(sum((p - t) ** 2 for p, t in zip(params, target)))


hist = optimize(GradientDescent(lr=0.0005), rosenbrock, rosenbrock_gradient, [-1.2, 1.0], steps=50)
print("GD 50 steps: last", [round(x, 4) for x in hist[-1]],
      "f", round(rosenbrock(hist[-1]), 4),
      "dist", round(distance_to_minimum(hist[-1]), 4))


GD 50 steps: last [-1.0125, 1.0331] f 4.0565 dist 2.0128


## 6. 三人赛：从 $(-1.2,\,1.0)$ 走约 400 步

同一起点、同一解析梯度。GD / 动量用源文件里偏保守的 lr（太大容易沿陡壁飞出去）；Adam 在这个弯曲谷上可以用更大的 lr，几百步内通常明显领先。

短程比较里默认 `Adam(lr=0.01)` 往往还没钻进谷；`lr=0.1` 才算 Rosenbrock 上比较像样的步长。


In [7]:
start = [-1.2, 1.0]
steps = 400
configs = [
    ("Gradient Descent", GradientDescent(lr=0.0005)),
    ("SGD + Momentum", SGDMomentum(lr=0.0001, momentum=0.9)),
    ("Adam", Adam(lr=0.1)),
]

print(f"start {start}, f={rosenbrock(start):.4f}, steps={steps}")
print(f"{'method':<18} {'x':>10} {'y':>10} {'f':>14} {'dist':>8}")
for name, opt in configs:
    hist = optimize(opt, rosenbrock, rosenbrock_gradient, start, steps=steps)
    final = hist[-1]
    print(
        f"{name:<18} {final[0]:10.4f} {final[1]:10.4f} "
        f"{rosenbrock(final):14.6f} {distance_to_minimum(final):8.4f}"
    )
print("target             1.0000     1.0000       0.000000   0.0000")
print("Adam 用像样的学习率（这里 0.1）通常最先靠近 (1,1)。")


start [-1.2, 1.0], f=24.2000, steps=400
method                      x          y              f     dist
Gradient Descent      -0.8624     0.7518       3.474960   1.8788
SGD + Momentum        -0.6159     0.3872       2.617322   1.7282
Adam                   0.7710     0.5937       0.052483   0.4664
target             1.0000     1.0000       0.000000   0.0000
Adam 用像样的学习率（这里 0.1）通常最先靠近 (1,1)。


## 对照表

| 函数 | 角色 |
|------|------|
| `rosenbrock` | 经典非凸测试函数，最小点 $(1,1)$ |
| `rosenbrock_gradient` | 解析 $\nabla f$，供 step 使用 |
| `GradientDescent.step` | $\theta\leftarrow\theta-\eta g$ |
| `SGDMomentum.step` | 速度缓冲，减轻振荡 |
| `Adam.step` | 一阶矩 + 二阶矩 + 偏差修正 |
| `optimize` | 固定步数循环；NaN 则停 |
| `distance_to_minimum` | 到 $(1,1)$ 的欧氏距离 |

要看完整 5000 步打印 demo（含学习率 / 动量 / 鞍点），运行：

```bash
python optimizers.py
```
